In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib
plt.rcParams.update({"text.usetex": True,
                     "axes.spines.right" : False,
                     "axes.spines.top" : False,
                     "font.size": 15,
                     "savefig.dpi": 400,
                     "savefig.bbox": 'tight',
                     'text.latex.preamble': r'\usepackage{amsfonts}'
                    }
                   )

from geomstats.geometry.euclidean import Euclidean
from geomstats.geometry.functions import SRVF, ProbabilityDistributions, L2Space
from geomstats.information_geometry.normal import NormalDistributions
from geomstats.geometry.discrete_curves import DiscreteCurves
import geomstats.backend as gs

import fdasrsf as fs

INFO: Using numpy backend


ValueError: numpy.ndarray size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
N_SAMPLES = 100
TARGET = [0,0.5]

lambda_ = np.linspace(-5,5,num=N_SAMPLES)

Rn = Euclidean(N_SAMPLES)
srvf = SRVF(lambda_)
L2 = L2Space(lambda_)
normal = NormalDistributions()
pdist = ProbabilityDistributions(lambda_)

def gaussian(mu,sig):
    scale = 1/(np.sqrt(2*np.pi)*sig)
    return scale*np.exp(-np.power(lambda_ - mu, 2.) / (2 * np.power(sig, 2.)))

yt = gaussian(*TARGET)

dRn = lambda xi,yi : -float(Rn.metric.dist(yi, yt))
dSRSF = lambda xi,yi : -srvf.metric.dist(yi, yt)   
dNormal = lambda xi,yi : -normal.metric.dist(gs.array([xi,0.5]), gs.array(TARGET))
dPD = lambda xi,yi : -pdist.metric.dist(yi, yt)   

R2 = Euclidean(2)
curves = DiscreteCurves(R2)

def dSRVF(i,yi):
    yi_aug = gs.array([lambda_, yi]).T
    yt_aug = gs.array([lambda_, yt]).T
    return -curves.square_root_velocity_metric.dist(yi_aug, yt_aug)

test_x = np.linspace(-5, 5, 51)
fig, axs = plt.subplots(1,5,figsize=(4*5, 4))

ground_truth_Rn = [dRn(i, gaussian(i,TARGET[1])) for i in test_x]
axs[0].plot(test_x, ground_truth_Rn , label=r'$\mathbb{R}^n$')

ground_truth_srsf = [dSRSF(i, gaussian(i,TARGET[1])) for i in test_x]
axs[1].plot(test_x, ground_truth_srsf, label='SRSF')

ground_truth_normaldist = [dNormal(i, gaussian(i,TARGET[1])) for i in test_x]
axs[2].plot(test_x, ground_truth_normaldist, label=r'$\mathcal{N}(\mu, \sigma)$')

ground_truth_pdist = [dPD(i, gaussian(i,TARGET[1])) for i in test_x]
axs[3].plot(test_x, ground_truth_pdist, label=r'$\mathcal{P}$')

ground_truth_srvf = [dSRVF(i, gaussian(i,TARGET[1])) for i in test_x]
axs[4].plot(test_x, ground_truth_srvf, label=r'SRVF')
fig.supxlabel(r'$\mathcal{X}$')
for ax in axs:
    ax.legend(ncol=2,loc='upper center',bbox_to_anchor=(0.45, 1.25))
plt.show()

In [ ]:
def amplitude_phase_distance(f1,f2):
    time = lambda_
    q1 = fs.f_to_srsf(f1, time)
    q2 = fs.f_to_srsf(f2, time)
    if np.allclose(q1, q2, atol=1e-1):
        return 0,0
    gam = fs.optimum_reparam(q1, time, q2, "DP2", lam=5e-4)
    fw = fs.warp_f_gamma(time, f2, gam)
    qw = fs.warp_q_gamma(time, q2, gam)

    da = np.sqrt(np.trapz((qw - q1) ** 2, time))
    M = time.shape[0]
    t = np.linspace(0,1,M)
    psi = np.sqrt(np.gradient(gam,time))
    q1dotq2 = np.trapz(psi, t)
    if q1dotq2 > 1:
        q1dotq2 = 1
    elif q1dotq2 < -1:
        q1dotq2 = -1

    dp = np.real(np.arccos(q1dotq2))
    
    return da,dp

In [ ]:
def amplitude_phase_distance(point_a, point_b):
    curves = np.zeros((len(point_a), 2))
    curves[...,0] = point_a
    curves[...,1] = point_b
    obj = fs.fdawarp(curves, lambda_)
    obj.srsf_align(parallel=True, MaxItr=50)
    dp = fs.efda_distance_phase(obj.qn[...,0], obj.qn[...,1])
    da = fs.efda_distance(obj.qn[...,0], obj.qn[...,1])
    del obj
    return da, dp

In [ ]:
fig, ax = plt.subplots()
y0 = gaussian(3,0.5)
y1 = gaussian(-2,0.5)
y2 = gaussian(-3,0.5)
y3 = gaussian(-2,0.8)
ax.plot(lambda_, y0, lw=2.0)
ax.plot(lambda_, y1, color='k', lw=2.0)
ax.plot(lambda_, y2, ls='--', color='k', lw=2.0)
ax.plot(lambda_, y3, ls='dashdot', color='k', lw=2.0)
print('Euclidean/MSE distance : [%.2f, %.2f]'%(Rn.metric.dist(y0, y1), Rn.metric.dist(y0, y2)))
print('SRSF distance : [%.2f, %.2f]'%(srvf.metric.dist(y0, y1), srvf.metric.dist(y0, y2)))
print('Amplitude-Phase distance : [%.2f, %.2f]'%(sum(amplitude_phase_distance(y0, y1)), 
                                                 sum(amplitude_phase_distance(y0, y2))))
print('L2 distance : [%.2f, %.2f]'%(L2.metric.dist(y0, y1), L2.metric.dist(y0, y2)))
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel(r'$f(\lambda)$')
plt.savefig('../expts/flat_metrics.png')
plt.show()

In [ ]:
fig, ax = plt.subplots()
efda = np.asarray([amplitude_phase_distance(yt, gaussian(i,TARGET[1])) for i in test_x])
ax.plot(test_x, -efda[:,0], label = r'$d_{a}$')
ax.plot(test_x, -efda[:,1], label = r'$d_{p}$')
ax.plot(test_x, -efda.sum(axis=1), label = r'$d_{a}+d_{p}$')
ax.legend()
ax.set_xlabel(r'$\mathcal{X}$')
ax.set_ylabel(r'$d_{\mathcal{M}}(x,x_{t})$')
plt.savefig('../expts/apdist.png')
plt.show()